In [1]:
import os
from pathlib import Path
from dotenv import dotenv_values, load_dotenv

env_path = Path("/Users/daniel/code/lca-lc-foundations/.env")
file_values = dotenv_values(env_path)

load_dotenv(env_path, override=True)

loaded_key = os.environ.get("LANGSMITH_API_KEY", "")
file_key = file_values.get("LANGSMITH_API_KEY", "")

print("Using endpoint:", os.environ.get("LANGSMITH_ENDPOINT"))
print("Key matches .env:", loaded_key == file_key)

Using endpoint: https://apac.api.smith.langchain.com
Key matches .env: True


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

/var/folders/d3/rd_439mx5bn4lts65x7vw2vw0000gn/T/ipykernel_25349/2611793992.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [4]:
from langchain.tools import tool

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

sql_query.invoke("SELECT * FROM Artist LIMIT 10")

"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains'), (6, 'Antônio Carlos Jobim'), (7, 'Apocalyptica'), (8, 'Audioslave'), (9, 'BackBeat'), (10, 'Billy Cobham')]"

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[sql_query]
)

In [6]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?")

response = agent.invoke(
    {"messages": [question]}
)

In [7]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?", additional_kwargs={}, response_metadata={}, id='4f4cbd5b-415d-4267-be01-be86426a455c'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 428, 'prompt_tokens': 142, 'total_tokens': 570, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EYTVHBzS3idjorERAZH7TBycKrHFf', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a134af-5e0a-73f0-8623-6a96f53c958f-0', tool_calls=[{'name': 'sql_query', 'args': {'query': "SELECT artist_name, popularity FROM artists WHERE artist_name LIKE 'S%' ORDER BY popularity D

In [8]:
print(response["messages"][-3].tool_calls[0]['args']['query'])

SELECT a.Name AS ArtistName, COUNT(t.TrackId) AS TrackCount FROM Artist a JOIN Album al ON al.ArtistId = a.ArtistId JOIN Track t ON t.AlbumId = al.AlbumId WHERE a.Name LIKE 'S%' GROUP BY a.Name ORDER BY TrackCount DESC LIMIT 1;
